# Kairos: real-data 14B resource pilot

Two reviewed public repair examples. Uses the same native Transformers runner, adapter settings and padded sequence geometry as the final public100 study. This pilot establishes resource and resume mechanics only. No 100-example training or repair-improvement claim. Free GPU only; no paid upgrade, Drive mount, credentials or API. The historical synthetic run is never resumed.

Use a desktop browser that passes the download probe. Each training stage is a separate explicit dispatch; never Run all or automatically retry a stage.


In [ ]:
import hashlib,json,os,platform,shutil,zipfile
from pathlib import Path,PurePosixPath
from google.colab import files
ROOT=Path('/content/kairos14-real014-v2')
if platform.system()!='Linux':raise RuntimeError('Use a free Linux GPU runtime, not the Mac')
if ROOT.exists():raise RuntimeError('Existing owned run preserved; inspect it instead of restarting')
if shutil.disk_usage('/content').free<24*1024**3:raise RuntimeError('Need 20GiB scratch plus 4GiB reserve')
uploaded=files.upload()
if set(uploaded)!={'kairos14-real-input014.zip'}:raise RuntimeError('Upload only the reviewed input ZIP')
raw=uploaded['kairos14-real-input014.zip']
if len(raw)!=771054 or hashlib.sha256(raw).hexdigest()!='2b552de098e0e5c61185c0d914ea59d92798f94baaa5007453f87c90b84e4178':raise RuntimeError('Input archive differs')
ROOT.mkdir(mode=0o700)
archive=ROOT/'input.zip';archive.write_bytes(raw)
with zipfile.ZipFile(archive) as z:
 names=z.namelist()
 if len(names)!=len(set(names)) or sum(i.file_size for i in z.infolist())>16*1024**2:raise RuntimeError('Oversized or duplicate package')
 manifest=json.loads(z.read('manifest.json'))
 if set(names)!=set(manifest['files'])|{'manifest.json'}:raise RuntimeError('Package membership differs')
 for name,item in manifest['files'].items():
  p=PurePosixPath(name)
  if p.is_absolute() or '..' in p.parts or '\\' in name or name!=p.as_posix():raise RuntimeError('Unsafe package path')
  data=z.read(name)
  if len(data)!=item['bytes'] or hashlib.sha256(data).hexdigest()!=item['sha256']:raise RuntimeError('Package member differs')
  target=ROOT/'bundle'/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
print({'input_verified':True,'real_examples':2,'training_started':False})


## Verify downloads before installing or downloading weights

Download this tiny probe. Verify its actual local bytes, then enter the local SHA-256 below. Stop here if download does not arrive. Do not allocate model weights merely because the cloud prints a hash.


In [ ]:
probe=ROOT/'kairos14-download-probe.json'
probe.write_text('{"purpose":"Kairos download preflight","version":1}\n')
PROBE_SHA256=hashlib.sha256(probe.read_bytes()).hexdigest()
print({'probe_bytes':probe.stat().st_size,'cloud_probe_sha256':PROBE_SHA256})
files.download(str(probe))


In [ ]:
LOCAL_PROBE_SHA256=''  # Fill only after checking the received local file.
if LOCAL_PROBE_SHA256!=PROBE_SHA256:raise RuntimeError('Local download verification required before setup')
DOWNLOAD_PREFLIGHT_VERIFIED=True


## Install and freeze the native training environment

Preserve the provider Torch/CUDA installation. A pinned dependency conflict stops before weights. Installation and package metadata are retained.


In [ ]:
import subprocess,sys,venv,importlib.metadata
assert DOWNLOAD_PREFLIGHT_VERIFIED
(ROOT/'tmp').mkdir()
venv.EnvBuilder(system_site_packages=True,with_pip=False).create(ROOT/'venv')
PYTHON=ROOT/'venv/bin/python';SOURCE=ROOT/'bundle/source'
constraints=[]
for package in ['torch','torchvision','triton','numpy']:
 try:constraints.append(package+'=='+importlib.metadata.version(package))
 except importlib.metadata.PackageNotFoundError:pass
pins=['transformers==4.57.6','peft==0.21.2','accelerate==1.15.0','bitsandbytes==0.50.2','huggingface-hub==0.36.2','safetensors==0.8.0']
(ROOT/'constraints.txt').write_text('\n'.join(constraints+pins)+'\n')
ENV={**os.environ,'PYTHONDONTWRITEBYTECODE':'1','PYTHONPATH':str(SOURCE),'TMPDIR':str(ROOT/'tmp'),'PIP_CONFIG_FILE':'/dev/null','HF_HUB_DISABLE_IMPLICIT_TOKEN':'1','HF_HOME':str(ROOT/'hf-cache'),'TORCH_HOME':str(ROOT/'torch-cache'),'XDG_CACHE_HOME':str(ROOT/'cache'),'WANDB_DISABLED':'true'}
with (ROOT/'installation.log').open('x') as log:
 result=subprocess.run([sys.executable,'-m','pip','--python',str(PYTHON),'install','--no-cache-dir','--index-url','https://pypi.org/simple','-c',str(ROOT/'constraints.txt'),'-r',str(SOURCE/'requirements-api.lock'),*pins],stdout=log,stderr=subprocess.STDOUT,env=ENV,timeout=1200)
if result.returncode:raise RuntimeError('Installation failed; preserve log, do not dispatch training')
print('Pinned native environment prepared; no weights loaded yet.')


## Download and verify the pinned 14B NF4 checkpoint

No provider token, paid endpoint or smaller-model fallback. The model manifest records actual dependency versions and every local model file.


In [ ]:
prepare_model="import hashlib,json,importlib.metadata,shutil,os\nfrom pathlib import Path\nfrom huggingface_hub import snapshot_download\nimport torch\nroot=Path(os.environ['KAIROS_REAL_ROOT']);model=root/'model'\nif not torch.cuda.is_available() or torch.cuda.get_device_capability(0)<(7,5):raise RuntimeError('Compatible free CUDA GPU required')\nrepo='unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit';revision='ecfc63c083900ce4f19daad695e711f38ab98fcf'\nsnapshot_download(repo,revision=revision,local_dir=str(model),allow_patterns=['*.json','*.safetensors','*.model','*.txt','*.jinja'],token=False,max_workers=1)\nexpected={'model-00001-of-00002.safetensors':('8fbefb1b257e9443b3b817b655449f1c2884afefcad1a5a6c33f9097169e383e',4998641583),'model-00002-of-00002.safetensors':('d5c3d547462b12bac817a5af131be0862220b67ec7a7114b07fdf810df6ac285',4933695760)}\ndef sha(p):\n with p.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()\nfor name,(digest,size) in expected.items():\n p=model/name\n if p.stat().st_size!=size or sha(p)!=digest:raise RuntimeError('Pinned weight bytes differ')\ncache=model/'.cache'\nif cache.exists():\n (root/'download-metadata.json').write_text(json.dumps({p.relative_to(model).as_posix():{'sha256':sha(p),'bytes':p.stat().st_size} for p in cache.rglob('*') if p.is_file()},sort_keys=True))\n shutil.rmtree(cache)  # Newly owned downloader metadata only; model bytes remain unchanged.\nowned=sum(p.stat().st_size for p in root.rglob('*') if p.is_file() and not p.is_symlink())\nif owned>20*1024**3 or shutil.disk_usage(root).free<4*1024**3:raise RuntimeError('Cloud scratch guard failed')\nmanifest={'model':repo,'revision':revision,'quantization':'nf4-double-quant','training_environment':{n:importlib.metadata.version(n) for n in ['torch','transformers','peft','accelerate','bitsandbytes']},'files':{p.relative_to(model).as_posix():{'sha256':sha(p),'bytes':p.stat().st_size} for p in model.rglob('*') if p.is_file()}}\n(root/'model-manifest.json').write_text(json.dumps(manifest,sort_keys=True))\nprint(json.dumps({'model':repo,'device':torch.cuda.get_device_name(0),'manifest_sha256':sha(root/'model-manifest.json'),'training_started':False}))\n"
path=ROOT/'prepare-model.py';path.write_text(prepare_model)
with (ROOT/'model-prepare.log').open('x') as log:
 result=subprocess.run([str(PYTHON),'-u',str(path)],cwd=SOURCE,env={**ENV,'KAIROS_REAL_ROOT':str(ROOT)},stdout=log,stderr=subprocess.STDOUT,timeout=2400)
print((ROOT/'model-prepare.log').read_text()[-2500:])
if result.returncode:raise RuntimeError('Model preparation failed; preserve result')
MODEL_SHA256=hashlib.sha256((ROOT/'model-manifest.json').read_bytes()).hexdigest()


In [ ]:
def run_stage(name,stop=None,resume=None):
 output=ROOT/'runs'/name
 if output.exists():raise RuntimeError('Stage already dispatched; preserve it, never automatically retry')
 args=[str(PYTHON),'-u','-m','scripts.coding_train014','fit','--study','resource2','--profile','14B','--corpus',str(ROOT/'bundle/data/corpus.json'),'--corpus-sha256','cc543ad92608f01aadeb2773e61423015a477f1257500804379dcf3a6d277048','--admission-directory',str(ROOT/'bundle/data/admission'),'--admission-index-sha256','b89ff292ac05722ebff85ea3a5f704861095cc411c586e3046a058d4d29a4e93','--model-manifest',str(ROOT/'model-manifest.json'),'--model-manifest-sha256',MODEL_SHA256,'--model-directory',str(ROOT/'model'),'--output',str(output),'--scratch-root',str(ROOT),'--scratch-bytes',str(20*1024**3)]
 if stop:args+=['--stop-after',str(stop)]
 if resume:
  receipt=resume/'kairos-checkpoint.json'
  args+=['--resume-checkpoint',str(resume),'--resume-receipt-sha256',hashlib.sha256(receipt.read_bytes()).hexdigest()]
 log_path=ROOT/(name+'-training.log')
 with log_path.open('x') as log:
  result=subprocess.run(args,cwd=SOURCE,env=ENV,stdout=log,stderr=subprocess.STDOUT,timeout=7200)
 print(log_path.read_text()[-3500:])
 if result.returncode:raise RuntimeError('Stage failed; preserve receipts and export evidence')
 receipt=json.loads((output/'receipt.json').read_text())
 print({key:receipt[key] for key in ['completed_steps','trainable_parameters','vram_headroom_fraction','resource_fit_verified']})
 return receipt


## Full condition: 20 real-data optimizer updates

Run once. Inspect the actual memory-reserve result before the next condition. A failed reserve check ends this pilot; export the failure instead of spending more GPU time.


In [ ]:
full=run_stage('full')
if not full['resource_fit_verified']:raise RuntimeError('Resource reserve failed; do not run more conditions')


## Interrupted condition: stop at update 10

Run once after the full resource check passes. No full-model retraining or alternate checkpoint is substituted.


In [ ]:
assert full['resource_fit_verified']
partial=run_stage('interrupted',stop=10)


## Resume to update 20 and compare

The runner validates every checkpoint file before resume. The comparison checks the actual adapter, optimizer, scheduler and random-state digests; it does not accept a successful-looking summary alone.


In [ ]:
resumed=run_stage('resumed',resume=ROOT/'runs/interrupted/checkpoint-10')
result=subprocess.run([str(PYTHON),'-m','scripts.coding_resource014','--root',str(ROOT/'runs')],cwd=SOURCE,env=ENV,capture_output=True,text=True,timeout=120)
print(result.stdout);print(result.stderr[-1000:])
if result.returncode:raise RuntimeError('Pilot comparison failed; preserve complete evidence')


## Export once, including failures

Save receipts, logs, proofs, the complete interrupted checkpoint and final adapter. Exclude base weights, environment, caches and account state. Download and locally verify every manifest member before owned runtime cleanup and sign-out. Do not rerun export or overwrite an existing archive.


In [ ]:
export=ROOT/'kairos14-real-result014.zip'
if export.exists():raise RuntimeError('Existing export preserved; download that file instead of regenerating')
selected=[]
for folder in ['runs']:
 for p in (ROOT/folder).rglob('*'):
  if not p.is_file():continue
  name=p.relative_to(ROOT).as_posix()
  if '/interrupted/checkpoint-10/' in name or p.name in {'receipt.json','failure.json','metrics.jsonl','resource-proof.json','kairos-checkpoint.json','trainer_state.json','resource-acceptance.json'} or ('/resumed/checkpoint-20/' in name and p.name in {'adapter_model.safetensors','adapter_config.json'}):selected.append(p)
selected += [p for p in ROOT.iterdir() if p.is_file() and (p.suffix in {'.log','.json','.txt','.py'} or p.name=='input.zip')]
if not selected or any(p.is_symlink() for p in selected):raise RuntimeError('Missing or unsafe export inputs')
if sum(p.stat().st_size for p in selected)>192*1024**2:raise RuntimeError('Raw export over budget; preserve runtime for inspected packaging')
def file_sha(path):
 with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
manifest={'version':'coding-real-resource-export014-v1','files':{p.relative_to(ROOT).as_posix():{'bytes':p.stat().st_size,'sha256':file_sha(p)} for p in selected},'full_corpus_training_complete':False,'repair_effectiveness_verified':False,'contains_base_weights':False}
with zipfile.ZipFile(export,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in selected:z.write(p,p.relative_to(ROOT).as_posix())
 z.writestr('manifest.json',json.dumps(manifest,sort_keys=True))
print({'export_bytes':export.stat().st_size,'export_sha256':file_sha(export)})
if export.stat().st_size>96*1024**2:raise RuntimeError('Export exceeds reviewed local download allowance; preserve existing archive')
files.download(str(export))
